## ⚙️ Before Running — Manual Upload Setup

1. On your local machine, create a folder (e.g. `edgelegal-upload/`) containing:
   - `triage_model/`  ← the full package from `backend/app/triage_model/`
   - `train.json`
   - `dev.json`
   - `test.json`
2. On Kaggle, go to **Datasets → New Dataset**, upload that folder, and name the dataset **`edgelegal-src`**.
3. In this notebook, click **+ Add data** (top-right) → find your `edgelegal-src` dataset → add it.
4. Then run all cells top to bottom.


# EdgeLegal — Triage Classifier Training (Kaggle T4)

Trains `cross-encoder/nli-deberta-v3-base` on ContractNLI with:
- Span-based premise extraction (annotation evidence spans only)
- Weighted CrossEntropyLoss (class imbalance correction)
- Macro-F1 early stopping (patience=2)

**Label mapping** (aligned to checkpoint `id2label` from HuggingFace config.json):
```
0 -> Contradiction  (checkpoint slot 0: contradiction)
1 -> Entailment     (checkpoint slot 1: entailment)
2 -> NotMentioned   (checkpoint slot 2: neutral)
```

Checkpoint saved to `/kaggle/working/checkpoints/best_model.pt`.

> **No fabricated accuracy targets.** Metrics appear in `training_log.json` after the run.


In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('torch', 'transformers>=4.40.0', 'accelerate', 'scikit-learn',
    'sentencepiece', 'protobuf')
print('Dependencies installed.')


In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory
    print('VRAM    :', round(mem / 1e9, 1), 'GB')


In [ ]:
from pathlib import Path

DATA_DIR   = Path('/kaggle/input/datasets/pranavreddy1455/edgelegal-dataset')
TRAIN_JSON = DATA_DIR / 'train.json'
DEV_JSON   = DATA_DIR / 'dev.json'
TEST_JSON  = DATA_DIR / 'test.json'
OUTPUT_DIR = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_JSON.exists(), f'train.json not found at {TRAIN_JSON}'
assert DEV_JSON.exists(),   f'dev.json not found at {DEV_JSON}'
assert TEST_JSON.exists(),  f'test.json not found at {TEST_JSON}'
print('Data dir   :', DATA_DIR)
print('Output dir :', OUTPUT_DIR)


In [ ]:
# Copy triage_model package from the edgelegal-src dataset into /kaggle/working
import shutil, sys
from pathlib import Path

src  = Path('/kaggle/input/datasets/pranavreddy1455/edgelegal-dataset/triage_model')
dest = Path('/kaggle/working/triage_model')

assert src.exists(), f'triage_model not found at {src} — check your dataset upload.'
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
print('Copied package from', src)

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')


In [ ]:
import json
from collections import Counter

with open(TRAIN_JSON) as f:
    train_data = json.load(f)

counts = Counter()
for doc in train_data['documents']:
    for ann_set in doc['annotation_sets']:
        for hyp_id, ann in ann_set['annotations'].items():
            counts[ann['choice']] += 1

total = sum(counts.values())
print('Train label distribution:')
for label, n in sorted(counts.items(), key=lambda x: -x[1]):
    pct = round(n / total * 100, 1)
    print(f'  {label:<20}: {n:>5}  ({pct}%)')
print(f'  Total               : {total}')
print()
print('Class weights will up-weight Contradiction (true minority ~12%).')
print('Label order: Contradiction=0, Entailment=1, NotMentioned=2')


In [ ]:
from triage_model.train import TrainConfig, main as train_main

config = TrainConfig(
    train_path  = str(TRAIN_JSON),
    dev_path    = str(DEV_JSON),
    output_dir  = str(OUTPUT_DIR),
    batch_size  = 32,
    lr          = 2e-5,
    epochs      = 3,
    warmup_ratio = 0.06,
    weight_decay = 0.01,
    max_length  = 512,
    early_stopping_patience = 2,
    smoke       = False,
)

print('Starting full training run ...')
train_main(config)


In [ ]:
import json
log_path = OUTPUT_DIR / 'training_log.json'
with open(log_path) as f:
    log = json.load(f)

print('Best macro-F1 :', log['best_macro_f1'])
print('Checkpoint    :', log['best_checkpoint'])
print()
print('Per-epoch results:')
for ep in log['epochs']:
    print(f"  Epoch {ep['epoch']}: train_loss={ep['train_loss']:.4f} "
          f"val_loss={ep['val_loss']:.4f} macro_f1={ep['macro_f1']:.4f}")
print()
print('REMINDER: These are real numbers from this run.')
print('No accuracy target was promised or fabricated in advance.')


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from triage_model.model   import TriageModel
from triage_model.dataset import ContractNLIDataset
from triage_model.evaluate import evaluate, compute_calibration, _collect_predictions

tok = AutoTokenizer.from_pretrained('cross-encoder/nli-deberta-v3-base')
dev_ds = ContractNLIDataset(str(DEV_JSON), tok, max_length=512)
dev_loader = DataLoader(dev_ds, batch_size=32, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TriageModel()
model.load(checkpoint_path=OUTPUT_DIR / 'best_model.pt', device=device)

metrics = evaluate(model, dev_loader, device)
print(metrics)

true_labels, pred_labels, confidences = _collect_predictions(model, dev_loader, device)
correct = [t == p for t, p in zip(true_labels, pred_labels)]
ece = compute_calibration(confidences, correct)
print()
print(ece)


In [ ]:
from triage_model.evaluate import select_threshold

frozen_tau, sweep = select_threshold(model, dev_loader, device)
print()
print('Frozen threshold for agent routing:', frozen_tau)
print('Pass this value to confidence_check_node() in agent/nodes.py.')


In [ ]:
import os
print('Checkpoint files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    mb = round(f.stat().st_size / 1e6, 1)
    print(f'  {f.name:<35} {mb} MB')
print()
print('Download best_model.pt from the Kaggle Output tab.')
print('Then set TRIAGE_CHECKPOINT env var or copy to checkpoints/best_model.pt.')
